# Analysis: Failure-Detection Quality (Section 7.5 / Appendix detection table)

Reproduces the detection statistics in the paper's detection table: how reliably
the gating signals flag classes where expert-restricted aggregation loses to the
**best prediction-only benchmark**. Reads the main experiment records (per-class
F1 of the method and the three benchmarks) joined to the gated records (per-class
gate flags and which signal fired).

Definitions (matching the paper):
- A class is **lost** at margin tau when
  `F1(experts_weighted) < max_b F1(b) - tau`, where the benchmarks `b` are
  `all_majority`, `em`, and `sml` -- i.e. the method is beaten by the best of the
  three. (Beating all three is a win.)
- A class is **gated** when the gate fired for it; `fired` records which signals
  tripped (A1, A2-under, A2-over).
- **recall** = caught / all real losses (coverage).
- **specificity** = left-alone / all not-lost (restraint).
- **per-signal precision** = real losses among that signal's fires (trust).

Main and gated runs are joined on (size, trial) and verified identical via
`subsample_hash`. QNLI is excluded as a structural (A3) failure with no geometric
signature. tau is swept over {0, 0.02, 0.05, 0.10}.


In [ ]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict

EXPERIMENT_JSONL = "data/results/experiment_records.jsonl"
GATED_JSONL      = "data/results/gated_records.jsonl"

EXPERT_KEY     = "experts_weighted"
BENCHMARK_KEYS = ["all_majority", "em", "sml"]
TAUS           = [0.0, 0.02, 0.05, 0.10]
EXCLUDE_DATASETS = {"QNLI"}
MIN_SUPPORT    = 1

## Load and join

The loss reference (per-class F1 of the method and benchmarks) lives in the
experiment records; the gate flags and fired signals live in the gated records.
We key both by (dataset, size, trial) and confirm the subsample hashes match, so
each (record, class) carries both its loss label and its gate decision.

In [ ]:
def load_jsonl(path):
    out = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

exp_records = load_jsonl(EXPERIMENT_JSONL)
gated_records = load_jsonl(GATED_JSONL)

# Index gated records by (dataset, size, trial).
gated_by_key = {}
for g in gated_records:
    gated_by_key[(g["dataset"], g["size"], g["trial"])] = g

f1_of = lambda r, mth, c: r["per_class_f1"][mth][str(c)]
safe_div = lambda a, b: a / b if b else float("nan")
print(f"{len(exp_records)} experiment records, {len(gated_records)} gated records.")

## Tally with per-signal breakdown

For each tau, classify every (record, class) as TP/FN/FP/TN against the
best-benchmark loss label, and separately attribute fires to A1 and A2 so their
individual precision and specificity can be reported.

In [ ]:
def analyze(tau):
    rows = []
    matched = mism = 0
    by_dataset = defaultdict(lambda: defaultdict(int))
    for r in exp_records:
        ds = r["dataset"]
        if ds in EXCLUDE_DATASETS:
            continue
        key = (ds, r["size"], r["trial"])
        g = gated_by_key.get(key)
        if g is None:
            continue
        if r.get("seed") != g.get("seed"):
            mism += 1
            continue
        matched += 1

        pcf1 = r["per_class_f1"]
        present = [b for b in BENCHMARK_KEYS if b in pcf1]
        if EXPERT_KEY not in pcf1 or not present:
            continue
        gated_cls = {int(c): fired for c, fired in g.get("gated_classes", {}).items()}

        c = by_dataset[ds]
        n_classes = len(pcf1[EXPERT_KEY])
        for cl in range(n_classes):
            cl_str = str(cl)
            if cl_str not in pcf1[EXPERT_KEY]:
                continue
            f1e = pcf1[EXPERT_KEY][cl_str]
            best = max(pcf1[b][cl_str] for b in present)
            lost = f1e < best - tau
            fired = gated_cls.get(cl, None)
            gated = fired is not None
            a1 = bool(fired) and any(s.startswith("A1") for s in fired)
            a2 = bool(fired) and any(s.startswith("A2") for s in fired)

            if   lost and gated: c["TP"] += 1
            elif lost:           c["FN"] += 1
            elif gated:          c["FP"] += 1
            else:                c["TN"] += 1
            if lost: c["TP_A1"] += a1; c["TP_A2"] += a2
            else:    c["FP_A1"] += a1; c["FP_A2"] += a2

    rows = [{"dataset": ds, **counts} for ds, counts in by_dataset.items()]
    df = pd.DataFrame(rows)
    for col in ["TP","FN","TN","FP","TP_A1","TP_A2","FP_A1","FP_A2"]:
        if col not in df.columns:
            df[col] = 0
    return df.fillna(0), matched, mism

def summarize(c):
    lost, notlost = c["TP"] + c["FN"], c["TN"] + c["FP"]
    return {
        "lost": int(lost),
        "recall":  safe_div(c["TP"], lost),
        "specificity": safe_div(c["TN"], notlost),
        "A1_prec": safe_div(c["TP_A1"], c["TP_A1"] + c["FP_A1"]),
        "A1_spec": safe_div(notlost - c["FP_A1"], notlost),
        "A2_prec": safe_div(c["TP_A2"], c["TP_A2"] + c["FP_A2"]),
        "A2_spec": safe_div(notlost - c["FP_A2"], notlost),
    }

summary_rows = []
for tau in TAUS:
    df, matched, mism = analyze(tau)
    df["lost"] = df["TP"] + df["FN"]
    print(f"\n=== tau = {tau:.2f}  (matched {matched} pairs) ===")
    print(df[["dataset", "lost", "TP", "FN", "TN", "FP"]]
          .sort_values("dataset").to_string(index=False))
    c = {k: int(df[k].sum()) for k in ["TP","FN","TN","FP","TP_A1","TP_A2","FP_A1","FP_A2"]}
    summary_rows.append({"tau": tau, **summarize(c)})

print("\nPooled detection statistics across tau:")
print(pd.DataFrame(summary_rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## Reading the table

This matches the paper's detection table: the combined-signal recall rises with
tau (catching a larger share of *severe* losses) at high specificity, while A2
precision stays high and stable (the reliable trigger) and A1 precision decays
(kept as a warning only). Per-dataset breakdowns can be obtained from the `analyze`
DataFrame if needed.